In [1]:
import os
import time
import pickle
import langchain
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

/tmp/ipykernel_23587/3691686813.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import UnstructuredURLLoader
/home/surya/llm-projects/news_research_tool/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite",
    temperature=0,
    max_tokens=500
)

#### (1) Load data

In [4]:
loaders = UnstructuredURLLoader([
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load()
len(data)

2

#### (2) Split data to create chunks

In [5]:
text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# As data is of type documents we can directly use split_documents over split_text in order to get the chunks.
docs=text_splitter.split_documents(data)

In [6]:
len(docs)

19

In [7]:
docs[0].metadata

{'source': 'https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html'}

#### (3) Create embeddings for these chunks and save them to FAISS index

In [8]:
# Create the embeddings of the chunks using openAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",
    output_dimensionality=768
)
# Pass the documents and embeddings inorder to create FAISS vector index
vectorindex_google_ai = FAISS.from_documents(docs, embeddings)

In [9]:
# Storing vector index create in local

vectorindex_google_ai.save_local("vector_index")

In [10]:
vectorindex=FAISS.load_local(
    "vector_index",
    embeddings,
    allow_dangerous_deserialization=True
)

#### (4) Retrieve similar embeddings for a given question and call LLM to retrieve final answer

In [11]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using the provided context.

Context:
{context}

Question:
{question}
""")

chain = (
    {
        "context": vectorindex.as_retriever(),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
)

In [12]:
response=chain.invoke(
    "What is the price of Tiago iCNG?"
)

print(response.content)

/home/surya/llm-projects/news_research_tool/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3719: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


[{'type': 'text', 'text': 'Based on the provided context, the Tiago iCNG is priced between Rs 6.55 lakh and Rs 8.1 lakh.', 'extras': {'signature': 'El4KXAERTTIPyzlPulE8qMKggtbieGUCbPzm+ujcKflMak/ewI3khBLIO9o14B4JJvBf24gg++8ujXYkAimvG5p7e0z8p4WDaUXJ23t909tiHSIgZemPIUFRQCSumjVB'}}]
